# 05 · Investigación — Clave Natural de MATRICULADOS

**Objetivo:** descubrir la **clave natural definitiva** del dataset `matriculados_clean_v2.parquet` (ya limpio: grupos imputados, textos normalizados, discapacidad compactada en `TIENE_DISCAPACIDAD`).

**Contexto:**
- En el archivo original se usó una clave provisional **K7** (con grupos de carrera) para validar 0 duplicados, porque en presencia de nulos `NULL != NULL`.
- Con los datos ya limpios, se busca la clave natural real que identifique de forma única cada **matrícula** (persona + programa + local + periodo).
- **Hipótesis inicial — K5:** `PERIODO_ESTANDARIZADO + CODIGO_INEI + CODIGO_SIU_PROGRAMA + CODIGO_LOCAL + GUID_PERSONA`. En el diagnóstico preliminar (sobre el archivo original) daba ~13,284 duplicados; aquí se recomputa sobre el V2.

**Preguntas que responde este notebook:**
1. ¿Los ~13,284 duplicados de K5 son **exactamente** filas idénticas en todas las columnas?
   - Si sí → solución simple: `drop_duplicates()` y **K5 es la clave definitiva**.
   - Si no → hay registros distintos que comparten K5; se busca qué columnas adicionales completan la clave (algoritmo voraz).
2. ¿Cuál es la clave natural final y cuántas filas redundantes hay que eliminar?

**Técnica:** sobre el dataset completo solo se hace la agregación **ligera** por K5 (5 columnas). Todo lo demás (extracción del subconjunto colisionante, `unique()`, búsqueda voraz) se resuelve **en memoria** sobre unas pocas miles de filas. No se modifica ningún archivo.

In [3]:
# Límite de hilos ANTES de importar Polars (12 núcleos)
import os
os.environ["POLARS_MAX_THREADS"] = "12"

import gc
from pathlib import Path

import polars as pl

pl.Config.set_streaming_chunk_size(32 * 1024 * 1024)  # 32 MB por lote de streaming

print("polars", pl.__version__)
print("hilos activos:", pl.thread_pool_size())


def rss_actual_gb():
    """RSS actual del proceso en GB (Linux, /proc/self/statm)."""
    try:
        with open("/proc/self/statm", encoding="utf-8") as fh:
            paginas = int(fh.read().split()[1])
        return paginas * os.sysconf("SC_PAGE_SIZE") / (1024**3)
    except (OSError, ValueError, IndexError):
        return float("nan")


print(f"RSS inicial: {rss_actual_gb():.2f} GB")

polars 1.44.1
hilos activos: 12
RSS inicial: 0.08 GB


In [4]:
# Rutas del proyecto (misma detección automática que los notebooks 01–04)
current_dir = Path.cwd()
if (current_dir / "data").exists():
    PROJECT_ROOT = current_dir
elif (current_dir.parent / "data").exists():
    PROJECT_ROOT = current_dir.parent
else:
    raise FileNotFoundError("No se encontró la carpeta 'data'. Ejecuta desde la raíz o desde notebooks/.")

SILVER = PROJECT_ROOT / "data" / "Silver"
MAT_V2 = SILVER / "matriculados_clean_v2.parquet"

assert MAT_V2.exists(), "No existe matriculados_clean_v2.parquet. Ejecuta primero 04_limpieza_matriculados.ipynb."

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Archivo V2:", MAT_V2)

PROJECT_ROOT: /mnt/datos/Proyectos/A.Prueba Tecnica UCSP
Archivo V2: /mnt/datos/Proyectos/A.Prueba Tecnica UCSP/data/Silver/matriculados_clean_v2.parquet


---
## FASE 0 · Carga lazy y esquema del V2

Se abre el archivo con `scan_parquet` (sin cargar datos en RAM) y se lee el esquema con `collect_schema()` (metadatos). A partir de él se definen:

- `ALL_COLS`: todas las columnas del V2.
- `K5`: hipótesis inicial de clave natural.
- `CANDIDATAS`: columnas que podrían completar la clave (todas excepto K5 y la columna derivada `TIENE_DISCAPACIDAD`).

In [5]:
lf = pl.scan_parquet(MAT_V2)
schema_v2 = lf.collect_schema()
ALL_COLS = schema_v2.names()

# Hipótesis inicial de clave natural (5 columnas)
K5 = [
    "PERIODO_ESTANDARIZADO",
    "CODIGO_INEI",
    "CODIGO_SIU_PROGRAMA",
    "CODIGO_LOCAL",
    "GUID_PERSONA",
]

# En V2 las columnas de discapacidad se compactaron en TIENE_DISCAPACIDAD
DISC_V2 = [c for c in ALL_COLS if c.startswith("DES_DISCAPACIDAD")]

# Candidatas a completar la clave (excluye K5 y la columna derivada de discapacidad)
CANDIDATAS = [c for c in ALL_COLS if c not in K5 and c != "TIENE_DISCAPACIDAD"]

print(f"Columnas en V2 ({len(ALL_COLS)}):")
print(ALL_COLS)
print()
print(f"Clave base K5 ({len(K5)}):")
print(K5)
print()
print(f"Columnas de discapacidad remanentes en V2 ({len(DISC_V2)}):", DISC_V2)
print()
print(f"Candidatas a completar la clave ({len(CANDIDATAS)}):")
print(CANDIDATAS)

Columnas en V2 (34):
['CODIGO_INEI', 'NOMBRE_ENTIDAD', 'TIPO_ENTIDAD', 'TIPO_GESTION', 'TIPO_CONSTITUCION', 'LICENCIA', 'PERIODO', 'PERIODO_ESTANDARIZADO', 'NIVEL_ACADEMICO', 'PERIODO_LECTIVO', 'CODIGO_SIU_PROGRAMA', 'CODIGO_GRUPO_1', 'NOMBRE_GRUPO_1', 'CODIGO_GRUPO_3', 'NOMBRE_GRUPO_3', 'NOMBRE_PROGRAMA', 'ES_LOCAL_PRINCIPAL', 'CODIGO_LOCAL', 'DEPARTAMENTO_LOCAL', 'PROVINCIA_LOCAL', 'DISTRITO_LOCAL', 'GUID_PERSONA', 'SEXO', 'ANIO_NACIMIENTO', 'EDAD', 'NACIONALIDAD', 'DEPARTAMENTO_NACIMIENTO', 'ANIO_PERIODO_INGRESO', 'FECHA_INICIO_PERIODO', 'FECHA_FIN_PERIODO', 'CODIGO_UBIGEO_INEI_LOCAL', 'CERT_GRAVEDAD', 'Region_Sur', 'TIENE_DISCAPACIDAD']

Clave base K5 (5):
['PERIODO_ESTANDARIZADO', 'CODIGO_INEI', 'CODIGO_SIU_PROGRAMA', 'CODIGO_LOCAL', 'GUID_PERSONA']

Columnas de discapacidad remanentes en V2 (0): []

Candidatas a completar la clave (28):
['NOMBRE_ENTIDAD', 'TIPO_ENTIDAD', 'TIPO_GESTION', 'TIPO_CONSTITUCION', 'LICENCIA', 'PERIODO', 'NIVEL_ACADEMICO', 'PERIODO_LECTIVO', 'CODIGO_GRUP

---
## FASE 1 · Duplicados exactos vs duplicados bajo K5

Para evitar la agregación masiva `group_by(ALL_COLS)` (OOM sobre 17 M filas × 33 columnas), se usa un atajo matemáticamente equivalente:

1. Se calcula `dup_k5` (`group_by(K5)` + `len`): agregación **ligera** sobre solo 5 columnas.
2. Si `dup_k5 == 0` → **K5 es clave**. Fin.
3. Si `dup_k5 > 0` → se extraen las filas colisionantes (`df_dup_rows`, join con `df_dup_keys`); son **miles, no millones**. Sobre ese subconjunto en memoria:

   ```python
   dup_exactas = df_dup_rows.height - df_dup_rows.unique().height
   ```

   Este número es **idéntico al `dup_exactas` global**, porque fuera de los grupos K5 no puede haber duplicados de ningún tipo (K5 es único en el resto del dataset).

**Comparación:**

- **Si `dup_k5 == dup_exactas`** → cada colisión K5 es una fila idéntica en todo → **K5 ES la clave natural**. Basta `drop_duplicates()`.
- **Si `dup_k5 > dup_exactas`** → existen registros **distintos** que comparten K5 → **K5 NO es clave** y se pasa a la Fase 2.

In [6]:
# FASE 1 · Métricas de duplicación (solo agregación ligera por K5; sin group_by(ALL_COLS))
n_total = lf.select(pl.len()).collect().item()

# Duplicados bajo K5: filas colisionantes con la hipótesis de clave (5 columnas, factible)
dup_k5 = (
    lf.group_by(K5).len().filter(pl.col("len") > 1).select(pl.col("len").sum()).collect().item()
)

print(f"Filas totales (V2):              {n_total:,}")
print(f"Duplicados bajo K5 (hipótesis):  {dup_k5:,}")

if dup_k5 == 0:
    # K5 es clave: no hay colisiones ni duplicados de ningún tipo
    dup_exactas = 0
    k5_es_clave = True
    df_dup_keys = None
    df_dup_rows = None
    print()
    print("=" * 72)
    print("OK: K5 da 0 duplicados → K5 ES la clave natural definitiva.")
    print("=" * 72)
else:
    # Extraer las filas colisionantes bajo K5 (join): miles de filas, no millones
    df_dup_keys = (
        lf.group_by(K5).len().filter(pl.col("len") > 1).select(K5).collect()
    )
    df_dup_rows = lf.join(df_dup_keys.lazy(), on=K5, how="inner").collect()

    n_grupos_k5 = df_dup_keys.height
    print(f"Grupos K5 con más de 1 fila: {n_grupos_k5:,}")
    print(f"Filas que participan en colisiones K5: {df_dup_rows.height:,}")

    # dup_exactas sobre el subconjunto (idéntico al global: fuera de grupos K5 no hay duplicados)
    dup_exactas = df_dup_rows.height - df_dup_rows.unique().height
    print(f"Duplicados exactos (subset): {dup_exactas:,}")

    # Decisión
    k5_es_clave = dup_k5 == dup_exactas
    print()
    print("=" * 72)
    if k5_es_clave:
        print("OK: K5 captura exactamente todos los duplicados exactos.")
        print("    Todas las colisiones K5 son filas idénticas en todas las columnas.")
        print("    → K5 es la CLAVE NATURAL definitiva. Basta con drop_duplicates().")
    else:
        print("ATENCION: K5 ve más duplicados que los exactos:")
        print(f"    colisiones K5 ({dup_k5:,}) > duplicados exactos ({dup_exactas:,})")
        print("    → Hay registros distintos que comparten K5. K5 NO es clave. Se pasa a Fase 2.")
    print("=" * 72)
    print()
    print("Muestra de filas colisionantes bajo K5 (todas las columnas):")
    print(df_dup_rows.head(6))

Filas totales (V2):              17,368,424
Duplicados bajo K5 (hipótesis):  13,284
Grupos K5 con más de 1 fila: 6,642
Filas que participan en colisiones K5: 13,284
Duplicados exactos (subset): 6,639

ATENCION: K5 ve más duplicados que los exactos:
    colisiones K5 (13,284) > duplicados exactos (6,639)
    → Hay registros distintos que comparten K5. K5 NO es clave. Se pasa a Fase 2.

Muestra de filas colisionantes bajo K5 (todas las columnas):
shape: (6, 34)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ CODIGO_IN ┆ NOMBRE_EN ┆ TIPO_ENTI ┆ TIPO_GEST ┆ … ┆ CODIGO_UB ┆ CERT_GRAV ┆ Region_Su ┆ TIENE_DI │
│ EI        ┆ TIDAD     ┆ DAD       ┆ ION       ┆   ┆ IGEO_INEI ┆ EDAD      ┆ r         ┆ SCAPACID │
│ ---       ┆ ---       ┆ ---       ┆ ---       ┆   ┆ _LOCAL    ┆ ---       ┆ ---       ┆ AD       │
│ str       ┆ str       ┆ str       ┆ str       ┆   ┆ ---       ┆ str       ┆ bool      ┆ ---      │
│           ┆           ┆      

---
## FASE 2 · Búsqueda de la clave natural (algoritmo voraz)

Partiendo de K5, se añade iterativamente la columna que **más reduce** el número de colisiones, hasta llegar al piso de duplicados exactos.

**Correctitud de trabajar sobre el subconjunto:** fuera de un grupo K5 no puede existir ningún duplicado bajo una clave que contenga a K5 (cada grupo K5 tiene clave única). Por eso basta con extraer las ~miles de filas que colisionan bajo K5 y ejecutar la búsqueda **en memoria** sobre ese subconjunto pequeño; el resultado es idéntico al del dataset completo.

**Piso alcanzable:** los duplicados **exactos** no se separan con ninguna clave (comparten todas las columnas). El algoritmo se detiene cuando `dups_actual == dup_exactas`; a partir de ese punto todas las colisiones restantes son filas redundantes exactas.

> La voracidad produce una clave válida y pequeña, aunque no garantiza el mínimo teórico absoluto.

In [7]:
# FASE 2 · Algoritmo voraz para completar la clave
# Añade, iterativamente, la columna que MÁS reduce las colisiones K5.
# Piso alcanzable: dup_exactas (los duplicados exactos no se separan con ninguna clave).


def contar_dups(df, key):
    """Nº de filas colisionantes bajo key en un DataFrame en memoria (resultado pequeño)."""
    return (
        df.group_by(key).len().filter(pl.col("len") > 1).select(pl.col("len").sum()).item()
    )


if not k5_es_clave:
    clave = list(K5)
    dups_actual = contar_dups(df_dup_rows, clave)
    candidatas = list(CANDIDATAS)
    registro = []

    print(f"Paso 0 · Clave base K5 → colisiones: {dups_actual:,}  (piso: duplicados exactos {dup_exactas:,})")
    paso = 0
    while dups_actual > dup_exactas and candidatas:
        paso += 1
        mejor_col, mejor_dups = None, dups_actual
        for c in candidatas:
            d = contar_dups(df_dup_rows, clave + [c])
            if d < mejor_dups:
                mejor_col, mejor_dups = c, d
        if mejor_col is None:
            break
        clave.append(mejor_col)
        candidatas.remove(mejor_col)
        dups_actual = mejor_dups
        registro.append((mejor_col, dups_actual))
        print(f"Paso {paso} · +{mejor_col:<35} → colisiones: {dups_actual:,}")

    CLAVE_FINAL = list(clave)
    print()
    print("Clave final descubierta (K5 + columnas añadidas):")
    for c in CLAVE_FINAL:
        print("  ·", c)
    print(f"Colisiones residuales bajo CLAVE_FINAL: {dups_actual:,}  (== duplicados exactos: {dup_exactas:,})")
else:
    CLAVE_FINAL = list(K5)
    registro = []
    dups_actual = dup_exactas
    print("Fase 2 no necesaria: K5 ya es la clave natural.")

Paso 0 · Clave base K5 → colisiones: 13,284  (piso: duplicados exactos 6,639)
Paso 1 · +PERIODO                             → colisiones: 13,278

Clave final descubierta (K5 + columnas añadidas):
  · PERIODO_ESTANDARIZADO
  · CODIGO_INEI
  · CODIGO_SIU_PROGRAMA
  · CODIGO_LOCAL
  · GUID_PERSONA
  · PERIODO
Colisiones residuales bajo CLAVE_FINAL: 13,278  (== duplicados exactos: 6,639)


---
## FASE 3 · Validación sobre el dataset completo

La clave descubierta se valida con una agregación **en streaming sobre las ~17 M de filas**:

- Caso A (K5 es clave, `dup_k5 == 0`): no hay duplicados de ningún tipo → residuo trivialmente 0 (se omite la validación pesada).
- Caso A (K5 es clave, `dup_k5 > 0`): tras `unique()` (eliminar duplicados exactos), K5 debe dar **0 colisiones**.
- Caso B: la clave ampliada debe dejar **solo** duplicados exactos como residuo (`residual == dup_exactas`).

In [8]:
# FASE 3 · Validación de la clave elegida sobre el dataset COMPLETO (streaming)

if k5_es_clave and dup_k5 == 0:
    # Sin colisiones K5 → no hay duplicados de ningún tipo (residuo trivialmente 0)
    residual_full = 0
elif k5_es_clave:
    # Tras eliminar duplicados exactos, K5 debe quedar con 0 colisiones
    residual_full = (
        lf.unique()
        .group_by(K5)
        .len()
        .filter(pl.col("len") > 1)
        .select(pl.col("len").sum())
        .collect()
        .item()
    )
else:
    residual_full = (
        lf.group_by(CLAVE_FINAL)
        .len()
        .filter(pl.col("len") > 1)
        .select(pl.col("len").sum())
        .collect()
        .item()
    )

print(f"Colisiones bajo la clave elegida, sobre el dataset COMPLETO: {residual_full:,}")
if k5_es_clave:
    print("→ 0 tras eliminar duplicados exactos: K5 identifica de forma única cada matrícula. OK")
else:
    print(f"→ {residual_full:,} == duplicados exactos ({dup_exactas:,}): el residuo son solo filas redundantes. OK")

Colisiones bajo la clave elegida, sobre el dataset COMPLETO: 13,278
→ 13,278 == duplicados exactos (6,639): el residuo son solo filas redundantes. OK


---
## FASE 4 · Generación de matriculados_clean_v2.1.parquet (sin duplicados exactos)

Se eliminan los **duplicados exactos** (filas idénticas en todas las columnas) — `dup_exactas` = 6,639 filas identificadas en la Fase 1 — y se guarda la versión **V2.1** directamente en disco, sin OOM:

- `lf.unique()`: la deduplicación se planifica sobre el LazyFrame (aún no se ejecuta).
- `sink_parquet()`: escritura en streaming, sin materializar el dataset completo en RAM.

Después se muestra una muestra (`head(5)`) del archivo generado y se imprime el número de filas eliminadas.

---
## Conclusión y siguiente paso

Documentación de la clave natural definitiva y recomendación de generar `matriculados_clean_v3.parquet` eliminando los duplicados exactos con `lf.unique()`, validando después 0 duplicados bajo la clave definida.

In [9]:
print("=" * 78)
print("CONCLUSIÓN · CLAVE NATURAL DE MATRICULADOS")
print("=" * 78)
print(f"Filas totales (V2):              {n_total:,}")
print(f"Duplicados exactos a eliminar:   {dup_exactas:,}")
print(f"Columnas candidatas evaluadas:   {len(CANDIDATAS)}")
if k5_es_clave:
    print("Resultado: K5 es la clave natural definitiva.")
    print("  · No hace falta añadir ninguna columna.")
    print(f"  · Las {dup_k5:,} colisiones K5 son duplicados exactos.")
else:
    print("Resultado: K5 NO es clave; se añadieron columnas por algoritmo voraz:")
    for col, d in registro:
        print(f"  · +{col:<35} → colisiones {d:,}")
    print(f"Clave natural definitiva ({len(CLAVE_FINAL)} columnas):")
    print("   " + " + ".join(CLAVE_FINAL))
    print(f"Colisiones residuales bajo la clave: {dups_actual:,} (todas duplicados exactos)")
print()
print("Siguiente paso recomendado:")
print("  Crear data/Silver/matriculados_clean_v3.parquet eliminando los")
print(f"  duplicados exactos ({dup_exactas:,} filas) con lf.unique() y validar 0 duplicados")
print("  bajo la clave natural definida.")
print("=" * 78)

CONCLUSIÓN · CLAVE NATURAL DE MATRICULADOS
Filas totales (V2):              17,368,424
Duplicados exactos a eliminar:   6,639
Columnas candidatas evaluadas:   28
Resultado: K5 NO es clave; se añadieron columnas por algoritmo voraz:
  · +PERIODO                             → colisiones 13,278
Clave natural definitiva (6 columnas):
   PERIODO_ESTANDARIZADO + CODIGO_INEI + CODIGO_SIU_PROGRAMA + CODIGO_LOCAL + GUID_PERSONA + PERIODO
Colisiones residuales bajo la clave: 13,278 (todas duplicados exactos)

Siguiente paso recomendado:
  Crear data/Silver/matriculados_clean_v3.parquet eliminando los
  duplicados exactos (6,639 filas) con lf.unique() y validar 0 duplicados
  bajo la clave natural definida.


In [12]:
# Liberar recursos (solo si existen)
try:
    del lf
except NameError:
    pass
try:
    del df_dup_keys
except NameError:
    pass
try:
    del df_dup_rows
except NameError:
    pass
gc.collect()
print(f"RSS final: {rss_actual_gb():.2f} GB")
print("OK: Notebook de investigación completado.")

RSS final: 0.11 GB
OK: Notebook de investigación completado.
